# Test 1.1.2: Écoulement 1D monophasique avec perte de charge régulière

In [ ]:
# import all the packages
from trustutils.jupyter import run
from trustutils import plot, files as tf
from datetime import date

from string import Template 
import re
import numpy as np
import sympy as sp
import pandas as pd
import os
import math

import matplotlib.pyplot as plt
from IPython.display import display, Markdown as md

## Introduction

In [ ]:
#run.introduction("Ali Fadlallah")
display(md("- Validation made by: Ali Fadlallah"))
today = date.today().strftime('%d/%m/%Y')
display(md("- Report generated: $%s$"%(today)))


Ce document vise à tester la capacité des solveurs dans code de thermohydraulique à simuler le champ de pression et la vitesse axiale d’un écoulement axial de fluide quand il est soumis à un frottement régulier. Plus largement, il s'inscrit dans le cadre d'une campagne de vérification des discrétisations/solveurs disponibles dans le code de thermohydraulique. L'objectif de cette vérification est de comparer les solutions numériques aux solutions analytiques. Nous nous concentrons ici sur le test 1.1.2 de cette campagne, pour lequel nous disposons d'une solution analytique.

Le contenu de ce document est structuré comme suit :



- Introduction
- Description du cas
- Solutions numériques
- Solutions analytiques 
- Résultats
- Évaluation de l'erreur
- Conclusion

In [ ]:
#run.TRUST_parameters()

## Description du cas
Dans ce cas, nous cherchons à vérifier que le code peut simuler avec précision les effets de le frottement régulier (perte de charge). Pour cela, nous créons un cas où nous isolons et mettons en évidence la force de frottement régulièr.

Ce cas simule un écoulement monophasique dans un canal avec une force de frottement régulier. Les parois sont adiabatiques et le fluide utilisé est de l’eau, considéré comme visqueuse. Les frottements sont modélisés par un coefficient de frottement constant, représentant des pertes de charge régulières.

Les conditions aux limites à l’entrée sont de type vitesse et enthalpie, et on impose une pression de sortie.

On réalise ce test avec le terme source "Perte_Charge_Reguliere". La loi d’état utilisée pour l’eau est "Loi_StiffenedGas".

Les dimensions du canal sont indiquées ci-dessous.



### Données géométriques

| Géométrie | Valeur
|:----------|:--------
| Longueur X [m]| 1.4 
| Largeur Y [m]| 1.4
| Hauteur Z  [m]| 4.2
| Diamètre hydraulique Dh [m]| 1.4

### Maillage

| Mailles | Valeur
|:------------|:----------
| nb mailles X | 1 
| dx [m] | 1.4 
|nb mailles Y | 1
| dy [m]| 1.4 
| nb mailles Z | 50
| dz [m]| 0.084

### Données physiques 

| Fluide | Eau
|:----------|:----------
| coefficient de frottement $C_f$ | 100.  
| Gravité [m.s$^{-2}$] | (0, 0, 0) 
| Densité [kg.m$^{-3}$] | 716

### Conditions limites 

| Entrée | Valeur 
| :---------------------|:------------------------
| Enthalpie liquide [J/kg]| 1.3e+06
| Enthalpie vapeur [J/kg]| 2.6e+06
| Vitesse liquide [m.s$^{-1}$] | (0, 0, 1)
| Vitesse vapeur [m.s$^{-1}$]| (0, 0, 0)


| Sortie | Valeur
| :---------------|:------------
| Pression [Pa] | 155e+05


### Schémas numériques testés


On teste 6 schémas en alternant entre deux types de discrétisations (VDF/PolyMAC) et deux types de solveurs (jacobienne_complete/solveur_pression), et l'option_VDF (P_imposee_aux_faces oui/non) :


| Cases          | Discrétisation  | Solveur             | option_VDF |
|:---------------|:----------------|:--------------------|:-----------|
| PolyMAC_jc     | PolyMAC         | jacobienne_complete | -          |
| PolyMac_sp     | PolyMAC         | solveur_pression    | -          |
| VDF_jc         | VDF             | jacobienne_complete | non        |
| VDF_jc_pface   | VDF             | jacobienne_complete | oui        |
| VDF_sp         | VDF             | solveur_pression    | non        |
| VDF_sp_pface   | VDF             | solveur_pression    | oui        |



**Remarque :** l'option_VDF est uniquement applicable à la discrétisation VDF.

## Solutions numériques


In [ ]:
# Cell to excute the cases
CF = 100.
Dh = 1.4

cases = {
        "VDF_jc_pface":("VDF", "jacobienne_complete", "OPTION_VDF { P_imposee_aux_faces oui }",
                        f"Perte_Charge_Reguliere longitudinale Z CF {CF} channel_1"),
        "VDF_sp_pface":("VDF", "solveur_pression", "OPTION_VDF { P_imposee_aux_faces oui }",
                        f"Perte_Charge_Reguliere longitudinale Z CF {CF} channel_1"),
        "VDF_jc":("VDF", "jacobienne_complete", "OPTION_VDF { P_imposee_aux_faces non }",
                        f"Perte_Charge_Reguliere longitudinale Z CF {CF} channel_1"),
        "VDF_sp":("VDF", "jacobienne_complete", "OPTION_VDF { P_imposee_aux_faces non }",
                        f"Perte_Charge_Reguliere longitudinale Z CF {CF} channel_1"),
        "PolyMAC_jc":("polyedriser dom polyMAC", "jacobienne_complete", "",
                      f"Perte_Charge_Reguliere_Isotrope_Directionnelle {{ CF {CF} }}"),
        "PolyMAC_sp":("polyedriser dom polyMAC", "solveur_pression", "",
                      f"Perte_Charge_Reguliere_Isotrope_Directionnelle {{ CF {CF} }}"),   
        }

run.reset()
run.defaultSuite_ = run.TRUSTSuite(runPrepare=False)

for case, (dis, solv, opt_vdf, frottement) in cases.items():
    run.addCaseFromTemplate(f"jdd.data",f"{case}",
                                {"option_vdf" : opt_vdf,
                                 "solveur_choisi" : solv,
                                 "discretisation_choisie" : dis,
                                 "frottement": frottement,
                                 "Dh": Dh
                                 })

#run.printCases()
run.runCases()
#run.tablePerf()

In [ ]:
def GetSonData(path):
    if not os.path.exists(path):
        raise ValueError(f"The path ({path}) does not exist!")
    # On plot le dernier instant
    donne = tf.SonSEGFile(path, None)
    # On recupere le nom des variables, leur dimension et la coordonnee que l'on veut tracer
    ncompo = donne.getnCompo()
    entries = donne.getEntries()
    t = donne.getValues(entries[0])[0]
    ## find closest value of t
    idx = -1
    X = donne.getXAxis()
    Y = np.zeros([len(entries[0::ncompo]), ncompo])
    for k in range(ncompo):
        for n, i in enumerate(entries[k::ncompo]):
            Y[n, k] = list(donne.getValues(i)[1])[idx]
    X = np.array(X)
    Y = np.array(Y[:, 0])
    return X, Y

# sondes if you have extra sondes thant you want to work with
# they should be in the jdd.data and you can add them to the vriables dictionnay 
# example you can add "vitesse":"SONDE_V"
variables = {"vitesse massique axiale":"SONDE_GM","pression":"SONDE_P", "Mach number": "SONDE_M",
            "vitesse axiale": "SONDE_V"}

num_data = {f"{case}": {} for case in cases.keys()}
''' 
a dictionnary to store all the numercial results in one place

a typical example of how the data will be stored:

num_ data = {'case_1':{'SONDE_GM': [],
                       'SONDE_P': [],
                       'SONDE_M': [],}
                           
            'case_2': etc..     
            }           
'''

for case in cases.keys():
    for variable, sonde in variables.items():
            path = os.path.join(f"{run.BUILD_DIRECTORY}/{case}/", f"jdd_{sonde}.son")  
            Z_values, data = GetSonData(path)
            num_data[case][variable] = data
                
#print("Numerical data successfully extracted.")

## Solutions analytiques

La solution analytique suivante concerne un écoulement incompressible soumis à une friction régulière de type $$\vec{F}=-\frac{C_f}{2D_h} \rho||u||\vec{u}$$

Les grandeurs d’intérêt de ce test sont la vitesse axiale et la pression, dont les évolutions le long du canal sont tracées ci-dessous.


\begin{equation*}
\begin{aligned}
    u(z) &= u_{liq}^{in},  \\
    P(z) &= P_{out} - \frac{C_f}{2D_h} \rho^{in}_{liq}{u^{in}_{liq}}^2(z-L), \\
\end{aligned}
\end{equation*}
avec :

$$
u_{liq}^{in} = 1 \; \text{m.s$^{-1}$}, \quad P_{out} = 155 \times 10^5 \; \text{Pa}, \quad C_{f} = 100,
$$
$$
D_{h} = 1.4 \; \text{m}, \quad L = 4.2 \; \text{m}, \quad \rho^{in}_{liq} = 716 \; \text{kg.m}^{-3}
$$

In [ ]:
# Analytical solutions

# initial conditions
L = 4.2 # m
P_sortie = 1.55e7 # Pawant to manipulate
V_entree =  1. # Kg.m/s
rho_entree = 716. # Kg/m3
# CF=500.  already defined
# Dh = 1.4 m

def vitesse_analytique(z):
    return np.array([1.0]*len(z))

def pression_analytique(z):
    return P_sortie - (0.5*CF/Dh)*(rho_entree * V_entree**2) * (z - L)

v_analyt =vitesse_analytique(Z_values)
p_analyt = pression_analytique(Z_values)

## Résultats

In [ ]:
def plot(cases):
    data_map = {
    "vitesse axiale": (v_analyt, r"m/s"),
    "pression": (p_analyt, "Pa")}
    
    
    plt.figure(figsize=(10, 5), dpi=1000)
    
    for i, variable in enumerate(data_map.keys()):
        plt.subplot(1, 2, i+1)
        plt.grid()
        

        sol_analyt = data_map[variable][0]
        plt.plot(Z_values, sol_analyt ,marker='*',label=f"solution analytique")
        plt.title(f"{variable}")
        
        for case in cases.keys():

            sol_num =  num_data[case][variable]

            plt.plot(Z_values, sol_num ,label=f"{case}")
            # disable scientific notation
            plt.ticklabel_format(useOffset=False,style='plain')
        
        # x and y label
        plt.xlabel('Z [m]')
        plt.ylabel(f"{variable} [{data_map[variable][1]}]")
        plt.legend()
    
    plt.tight_layout()
    plt.savefig("velocity_pressure_profile.pdf", dpi=1000)
    plt.show()
    
plot(cases)
#for case in cases.keys():
#    plot(case)

### Convergrences

In [ ]:
def extract_residu_newton(path_to_file):        
    # Initialize lists to store extracted data
    times = []
    newtons = []
    residus = []
    # Read the file line by line
    with open(path_to_file, 'r') as file:
        for line in file:
            # Match lines containing data (skip headers and footers)
            match = re.match(r"┃ t ([\d\.e\+\-]+)  │ Δt .* │ *(\d+) It │ ([\d\.e\+\-]+)", line)
            if match:
                time = float(match.group(1))
                newton = int(match.group(2))
                residu = float(match.group(3))
        
                times.append(time)
                newtons.append(newton)
                residus.append(residu)
                
    times = np.array(times)[1:]
    newtons = np.array(newtons)[1:]
    residus = np.array(residus)[1:]
    
    return times, newtons, residus

def plot_convergences(cases):
    
    plt.figure(figsize=(12, 6), dpi=400)
    
    plt.subplot(1, 2, 1)
    plt.grid()
    plt.title(f"Evolutions des résidus")
    plt.xlabel('Temps [s]')
    plt.ylabel("Résidu (max|Ri|)")

    for case in cases.keys():
        path = f"{run.BUILD_DIRECTORY}/{case}/jdd.list"
        time, _, residu = extract_residu_newton(path)
        plt.plot(time, residu, marker='x', label=f"{case}")

    plt.yscale('log')
    #plt.xscale('log')
    plt.legend()
    plt.tight_layout()
    
    
    # Newton iteration plot
    plt.subplot(1, 2, 2)
    plt.grid()
    plt.title(f"Evolutions des itérations Newton")
    plt.xlabel('Temps [s]')
    plt.ylabel("Nb des itérations Newton")

    for case in cases.keys():
        path = f"{run.BUILD_DIRECTORY}/{case}/jdd.list"
        time, newton, _ = extract_residu_newton(path)
        plt.plot(time, newton, marker='x', label=f"{case}")
            
    
    y_integer = range(int(plt.ylim()[0])+1, int(plt.ylim()[1])+1 )
    plt.yticks(y_integer)
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_convergences(cases)

## Évaluation des Erreurs

Les résultats obtenus avec le code sont en principe pour un écoulement compressible. Nous proposons donc de les comparer à la solution analytique incompressible monophasique en introduisant le nombre de Mach $M$, qui est le rapport entre la vitesse de l’écoulement et la vitesse du son dans le liquide, soit $\frac{u}{c}$. La solution compressible $(u, p)(\vec{x}, t)$ est proche de la solution incompressible $(u_*, p_*)(\vec{x}, t)$ comme suit :


\begin{equation*}
\begin{aligned}
    \left\{
    \begin{aligned}
    &u(\vec{x}, t) = u_*(\vec{x}, t) + \mathcal{O}(M)\\
    &p(\vec{x}, t) = p_*(\vec{x}, t) + \mathcal{O}(M^2)\\
    \end{aligned}\right.
\end{aligned}
\end{equation*}

Cela signifie que l’erreur relative entre la vitesse $u$ compressible (le code) et incompressible (solution analytique) sera proportionnelle à $M$, et celle de la pression $p$ à $M^2$.

On trouve une valeur constante de $M \approx 10^{-3}$ pour tous les cas testés (voir tableau ci-dessous) 

In [ ]:
# Mach number table
mach_nb = {f"{case}": max(num_data[case]["Mach number"]) for case in cases.keys()}
df_mach = pd.DataFrame(mach_nb, index=['Mach number'])
mach_table = df_mach.to_markdown(floatfmt=(".1g"))
display(md(mach_table))

Deux critères sont considérés pour que le test soit réussi.



- **Erreur relative:**  L’erreur relative entre la solution numérique calculée par le code et la solution analytique correspondante est calculée comme suit :

    $$ 
        \text{Erreur} \: \% = \text{max}\left| \frac{V_{\text{code}} - V_{\text{analytique}}}{V_{\text{analytique}} + \epsilon}\right| \times 100 
    $$

    où $V_{\text{code}}$ est la valeur calculée par le code, $V_{\text{analytique}}$ est la valeur analytique, et $\epsilon = 10^{-10}$ est utilisé pour éviter la division par zéro dans le cas où $V_{\text{analytique}} = 0$.
    
    Les critères de succès sont définis en fonction du nombre de Mach :
    
    - **Pression :** Le critère de succès est fixé à $100 \times M^2 = 10^{-4}$, soit $0.01\%$.
    - **Vitesse :** Le critère de succès est fixé à  $M = 10^{-3}$, soit $0.1\%$.
     

- **Ecart de variation P_in - P_out:** L’écart entre la variation $\text{P}_{\text{in}}-\text{P}_{\text{out}}$ calculée par le code et la solution analytique est calculé comme suit : 
    $$
    \Delta \text{P}_{\text{in$-$out}} = \left|\left[\frac{|\text{P}_{\text{in}} - \text{P}_{\text{out}}|}{\text{P}_{\text{in}}}\right]_{\text{code}} - \left[ \frac{|\text{P}_{\text{in}} - \text{P}_{\text{out}}|}{\text{P}_{\text{in}}} \right]_{\text{analytique}}\right| \times 100
    $$
    Le critère de succès est fixé à 1%.
    
    
Le tableau suivant présente les différentes erreurs pour chaque test case :

- $\color{green}{\checkmark}$ est affiché si l'erreur est acceptable
- $\color{orange}{\text{!}}$ est affiché si l'erreur est dans les 5 % du critère.
- $\color{red}{\times}$ est affiché si l'erreur dépasse le critère.


Une fiche est vérifiée, c’est à dire sans anomalie, si aucune ligne ne contient $\color{red}{\times}$ comme résultat.



In [ ]:
def calculate_ecart(data_code, data_analyt):
    """Calculates the maximum ecart between two data sets."""
    ecart = np.abs(data_code - data_analyt) * 100 / np.maximum(np.abs(data_analyt), 1e-10)
    return np.max(ecart)

def calculate_delta_p_in_out(data_code, data_analyt):
    """Calculates the delta P in-out ecart."""
    p_in_code, p_out_code = data_code[0], data_code[-1]
    p_in_analyt, p_out_analyt = data_analyt[0], data_analyt[-1]
    ecart_code = np.abs(p_in_code - p_out_code) * 100 / np.maximum(np.abs(p_in_code), 1e-10)
    ecart_analyt = np.abs(p_in_analyt - p_out_analyt) * 100 / np.maximum(np.abs(p_in_analyt), 1e-10)
    return np.abs(ecart_code - ecart_analyt)

def format_value(val, criteria):
    """Formats a value based on criteria."""
    if val <= criteria:
        symbol, color = r"\checkmark", "green"
    elif val <= criteria * 1.05:
        symbol, color = r"!", "orange"
    else:
        symbol, color = r"\times", "red"
    return rf"${val:.2g} \% \color{{{color}}}{{{symbol}}}$"

def create_dataframe(cases, num_data, Z_values, functions):
    """Creates a DataFrame with ecart values."""
    data = []
    for case in cases.keys():
        for variable, func in functions.items():
            if variable == r"$\Delta P_{in-out}$":
                ecart = calculate_delta_p_in_out(num_data[case]["pression"], func(Z_values))
            else:
                ecart = calculate_ecart(num_data[case][variable], func(Z_values))
            data.append({"case": case, "variable": variable, "ecart": ecart})
    #print(data)
    df = pd.DataFrame(data)
    df = df.pivot_table(index="case", columns="variable", values="ecart")
    return df

def format_dataframe(df, criteria):
    """Formats the DataFrame for display."""
    df_formatted = df.rename(columns={"pression": r"$\Delta P$",
                                     "vitesse axiale":r"$\Delta u$"})
    # Switch the first and second columns
    cols = df_formatted.columns.tolist()
    cols[0], cols[1] = cols[1], cols[0]
    
    df_formatted = df_formatted[cols]
    for col, crit in criteria.items():
        df_formatted[col] = df_formatted[col].apply(lambda x: format_value(x, crit))
    return df_formatted

def display_markdown_table(df):
    """Displays a DataFrame as a Markdown table."""
    df.insert (0, 'cases', df.index)
    fmt = ['---' for _ in df.columns]
    df_fmt = pd.DataFrame([fmt], columns=df.columns)
    df_formatted = pd.concat([df_fmt, df])
    display(md(df_formatted.to_csv(sep="|", index=False)))

functions = {
    "pression": pression_analytique,
    r"$\Delta P_{in-out}$": pression_analytique,
    "vitesse axiale": vitesse_analytique
}
criteria = {
    r"$\Delta P$": 0.01,
    r"$\Delta u$": 0.1,
    r"$\Delta P_{in-out}$": 1.,
}

df = create_dataframe(cases, num_data, Z_values, functions)
df_formatted = format_dataframe(df, criteria)
display_markdown_table(df_formatted)

In [ ]:
validated = []
not_validated = []
acceptable = []

for header in list(df)[:-1]:
    if header == r"$\Delta P$" : criteria = 0.01
    if header == r"$\Delta u$" : criteria = 0.1
    if header==r"$\Delta P_{in-out}$" : criteria = 1.

    errors = df.loc[:,header]
    if any(error > criteria*1.05 for error in errors):
        not_validated.append(header)
    elif any(criteria< error <= criteria*1.1 for error in errors):
        acceptable.append(header)
    else:
        validated.append(header)
        
        
# Output the Conclusion

if len(validated) == len(list(df)[:-1]):
    display(md("Les résultats sont satisfaisants."))
else:
    if len(validated)>0:
        display(md(f"Les résultats sont satisfaisants pour **{', '.join(validated)}**"))
    if len(acceptable)>0:
        display(md(f"Les résultats de **{', '.join(acceptable)}** sont acceptables puisque l’erreur\
               se situe à moins de 5 % de la critère choisie"))
    if len(not_validated)>0:
        display(md(f"Les résultats de **{', '.join(not_validated)}** ne sont pas acceptables."))
        
display(md("## Conclusion"))       
if len(not_validated) == 0:
    display(md("**Cette fiche est vérifiée avec succès.**"))
else:
    display(md(f"**Cette fiche n'est pas vérifiée pour {', '.join(not_validated)}.**"))